In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

# =========================
# 1. LOAD DATASET
# =========================
df = pd.read_csv("../data/processed/v2/development_token_nohref_in_text.csv")

# Target
y = df["label"]

# Features
X = df.drop(columns=["label", "Id"])

# =========================
# 2. DEFINE COLUMNS
# =========================
text_col = "text"
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# =========================
# 3. PIPELINE (baseline forte)
# =========================
preprocess = ColumnTransformer(
	transformers=[
		("text", TfidfVectorizer(
			max_features=50000,
			ngram_range=(1, 2),
			min_df=5
		), text_col),
		("num", StandardScaler(), numeric_cols)
	]
)

model = LogisticRegression(
	max_iter=1000,
	n_jobs=-1
)

pipeline = Pipeline([
	("prep", preprocess),
	("clf", model)
])

# =========================
# 4. OOF PREDICTIONS
# =========================
cv = StratifiedKFold(
	n_splits=5,
	shuffle=True,
	random_state=42
)

y_pred = cross_val_predict(
	pipeline,
	X,
	y,
	cv=cv,
	n_jobs=-1
)

# =========================
# 5. CONFUSION MATRIX
# =========================
cm = confusion_matrix(y, y_pred)

cm_df = pd.DataFrame(
	cm,
	index=[f"true_{i}" for i in range(cm.shape[0])],
	columns=[f"pred_{i}" for i in range(cm.shape[1])]
)

print("\n=== CONFUSION MATRIX (COUNTS) ===")
print(cm_df)

# =========================
# 6. NORMALIZED CONFUSION
# =========================
cm_norm = cm / cm.sum(axis=1, keepdims=True)

cm_norm_df = pd.DataFrame(
	np.round(cm_norm, 3),
	index=[f"true_{i}" for i in range(cm.shape[0])],
	columns=[f"pred_{i}" for i in range(cm.shape[1])]
)

print("\n=== CONFUSION MATRIX (ROW-NORMALIZED) ===")
print(cm_norm_df)

# =========================
# 7. MAJOR CONFUSIONS
# =========================
errors = []

for i in range(cm_norm.shape[0]):
	for j in range(cm_norm.shape[1]):
		if i != j and cm_norm[i, j] >= 0.10:
			errors.append((i, j, cm_norm[i, j]))

errors_df = pd.DataFrame(
	errors,
	columns=["true_label", "pred_label", "rate"]
).sort_values("rate", ascending=False)

print("\n=== MAJOR CONFUSIONS (rate >= 10%) ===")
print(errors_df)



=== CONFUSION MATRIX (COUNTS) ===
        pred_0  pred_1  pred_2  pred_3  pred_4  pred_5  pred_6
true_0   18074     841     747     896     392    2334     257
true_1     863    7694    1006     325      54     529     117
true_2    1083     798    8316     399      54     378     133
true_3    2247     653     671    4376     640    1186     204
true_4     420      60      66     273    7431     316       8
true_5    4780     942     604    1069     724    4625     309
true_6     526     168     116     186      31     281    1794

=== CONFUSION MATRIX (ROW-NORMALIZED) ===
        pred_0  pred_1  pred_2  pred_3  pred_4  pred_5  pred_6
true_0   0.768   0.036   0.032   0.038   0.017   0.099   0.011
true_1   0.082   0.727   0.095   0.031   0.005   0.050   0.011
true_2   0.097   0.071   0.745   0.036   0.005   0.034   0.012
true_3   0.225   0.065   0.067   0.439   0.064   0.119   0.020
true_4   0.049   0.007   0.008   0.032   0.867   0.037   0.001
true_5   0.366   0.072   0.046   0.082  

In [2]:
df.columns

Index(['Id', 'text', 'source', 'title', 'n_tokens', 'title_ratio', 'year',
       'month', 'has_timestamp', 'label', 'log_n_links', 'source_entropy',
       'source_max_prior', 'source_support'],
      dtype='object')

In [4]:
df_ts = df[df["has_timestamp"] == 1].copy()

print("Articoli con timestamp:", len(df_ts))
print("Percentuale:", len(df_ts) / len(df))


Articoli con timestamp: 52246
Percentuale: 0.6531076553827692


In [5]:
year_label_dist = (
	df_ts
	.groupby(["year", "label"])
	.size()
	.unstack(fill_value=0)
)

year_label_dist.head()


label,0,1,2,3,4,5,6
year,,,,,,,
2004.0,2636,1413,907,1873,1680,2227,507
2005.0,545,180,94,158,165,220,41
2006.0,3255,1319,1738,848,777,1104,274
2007.0,6837,3248,4730,2124,1581,3567,770
2008.0,2132,1142,1229,931,643,1017,334


In [6]:
year_label_prior = year_label_dist.div(
	year_label_dist.sum(axis=1),
	axis=0
)

year_label_prior.round(3)


label,0,1,2,3,4,5,6
year,,,,,,,
2004.0,0.234,0.126,0.081,0.167,0.149,0.198,0.045
2005.0,0.388,0.128,0.067,0.113,0.118,0.157,0.029
2006.0,0.349,0.142,0.187,0.091,0.083,0.119,0.029
2007.0,0.299,0.142,0.207,0.093,0.069,0.156,0.034
2008.0,0.287,0.154,0.165,0.125,0.087,0.137,0.045


In [7]:
import numpy as np

year_entropy = -(
	year_label_prior * np.log(year_label_prior + 1e-12)
).sum(axis=1)

year_entropy.describe()


count    5.000000
mean     1.778206
std      0.059804
min      1.703278
25%      1.739300
50%      1.773934
75%      1.827623
max      1.846893
dtype: float64

In [8]:
problem_labels = [0, 3, 5, 6]

year_label_prior[problem_labels].round(3)


label,0,3,5,6
year,,,,
2004.0,0.234,0.167,0.198,0.045
2005.0,0.388,0.113,0.157,0.029
2006.0,0.349,0.091,0.119,0.029
2007.0,0.299,0.093,0.156,0.034
2008.0,0.287,0.125,0.137,0.045


In [9]:
problem_labels = [0, 3, 5, 6]

month_label_dist = (
	df_ts
	.groupby(["month", "label"])
	.size()
	.unstack(fill_value=0)
)

month_label_prior = month_label_dist.div(
	month_label_dist.sum(axis=1),
	axis=0
)

month_label_prior[problem_labels].round(3)


label,0,3,5,6
month,,,,
1.0,0.298,0.112,0.151,0.042
2.0,0.293,0.108,0.134,0.035
3.0,0.319,0.090,0.137,0.025
5.0,0.314,0.074,0.144,0.040
6.0,0.298,0.087,0.145,0.028
7.0,0.330,0.101,0.159,0.032
8.0,0.312,0.131,0.141,0.043
9.0,0.312,0.109,0.158,0.028
10.0,0.258,0.146,0.173,0.043


In [10]:
import numpy as np

month_entropy = -(
	month_label_prior * np.log(month_label_prior + 1e-12)
).sum(axis=1)

month_entropy.describe()


count    11.000000
mean      1.795024
std       0.043216
min       1.738922
25%       1.758708
50%       1.789529
75%       1.821581
max       1.861987
dtype: float64

In [11]:
month_label_prior[problem_labels].var()


label
0    0.000692
3    0.000502
5    0.000282
6    0.000064
dtype: float64